# Ingestion Benchmark Notebook

Benchmarks the ingestion pipeline by processing each sample PDF individually and measuring:
- **Processing time** per document (wall clock, from orchestrator start to completion)
- **OCR token count** per document (extracted text measured with `tiktoken` using `cl100k_base`)
- Derived metrics: tokens/page, time/page, tokens/second

### Prerequisites
- Deployed Function App with working orchestrator endpoint
- `.env` file in this directory with `FUNCTION_URI`, `FUNCTION_KEY`, `STORAGE_ACCOUNT_NAME`
- Sample PDFs in `../sample_data/`
- `pip install python-dotenv requests pypdf tiktoken pandas azure-storage-blob azure-identity`

In [ ]:
import os
import glob
import json
import time
import requests
import pandas as pd
import tiktoken
from datetime import datetime
from pypdf import PdfReader
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import clear_output, display

load_dotenv(override=True)

FUNCTION_URI = os.getenv("FUNCTION_URI")
FUNCTION_KEY = os.getenv("FUNCTION_KEY")
STORAGE_ACCOUNT_NAME = os.getenv("STORAGE_ACCOUNT_NAME")

print(f"Function App: {FUNCTION_URI}")
print(f"Storage:      {STORAGE_ACCOUNT_NAME}")

## Step 1 — Discover Sample Documents

Scan `../sample_data/` for PDFs, count pages with `pypdf`, and display a manifest.

In [ ]:
sample_dir = os.path.join("..", "sample_data")
pdf_files = sorted(glob.glob(os.path.join(sample_dir, "*.pdf")))

manifest = []
for path in pdf_files:
    reader = PdfReader(path)
    manifest.append({
        "filename": os.path.basename(path),
        "local_path": path,
        "pages": len(reader.pages),
        "size_mb": round(os.path.getsize(path) / (1024 * 1024), 2)
    })

manifest_df = pd.DataFrame(manifest).sort_values("pages").reset_index(drop=True)
print(f"Found {len(manifest_df)} sample PDFs:\n")
display(manifest_df[["filename", "pages", "size_mb"]])

## Step 2 — Create Benchmark Index & Upload PDFs

Creates a fresh AI Search index and uploads all sample PDFs to blob storage under a unique benchmark prefix.

In [ ]:
# Settings
SOURCE_CONTAINER = "pdf-content"
EXTRACT_CONTAINER = "pdf-extract"
EMBEDDING_MODEL = "text-embedding-3-large"
BENCHMARK_ID = datetime.now().strftime("%Y%m%d%H%M%S")
BLOB_PREFIX = f"benchmark-{BENCHMARK_ID}"

# Helper: call Function App API
def call_function(path, body):
    uri = f"{FUNCTION_URI}/api/{path}?code={FUNCTION_KEY}"
    resp = requests.post(uri, json=body, timeout=120)
    resp.raise_for_status()
    return resp

# Create benchmark index
print("Creating benchmark index...")
resp = call_function("create_new_index", {
    "index_stem_name": f"bench-{BENCHMARK_ID[:8]}",
    "fields": {"content": "string", "pagenumber": "int", "sourcefile": "string",
               "sourcefilepath": "string", "sourcepage": "string", "category": "string"},
    "dimensions": 3072
})
INDEX_NAME = resp.text.strip().strip('"')
print(f"Index created: {INDEX_NAME}")

# Upload PDFs to blob storage
print(f"\nUploading {len(manifest_df)} PDFs to {SOURCE_CONTAINER}/{BLOB_PREFIX}/...")
blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net",
    credential=DefaultAzureCredential()
)
container_client = blob_service.get_container_client(SOURCE_CONTAINER)

for _, row in manifest_df.iterrows():
    blob_name = f"{BLOB_PREFIX}/{row['filename']}"
    with open(row["local_path"], "rb") as f:
        container_client.upload_blob(name=blob_name, data=f, overwrite=True)
    print(f"  Uploaded: {blob_name} ({row['size_mb']} MB, {row['pages']} pages)")

print("\nAll files uploaded.")

## Step 3 — Benchmark: Process Each Document Individually

Triggers `pdf_orchestrator` for each document **one at a time**, measuring wall-clock processing time from submission to completion.

In [ ]:
POLL_INTERVAL = 15  # seconds
TIMEOUT = 30 * 60   # 30 minutes per doc

results = []

for i, row in manifest_df.iterrows():
    blob_name = f"{BLOB_PREFIX}/{row['filename']}"
    print(f"\n[{i+1}/{len(manifest_df)}] Processing: {row['filename']} ({row['pages']} pages)")
    
    payload = {
        "source_container": SOURCE_CONTAINER,
        "extract_container": EXTRACT_CONTAINER,
        "prefix_path": blob_name,
        "index_name": INDEX_NAME,
        "automatically_delete": False,  # keep extracts for token counting
        "analyze_images": False,
        "chunking_strategy": "pagewise",
        "embedding_model": EMBEDDING_MODEL,
        "cosmos_logging": False,
        "entra_id": f"bench-{BENCHMARK_ID}",
        "session_id": f"bench-session-{i}"
    }
    
    # Start orchestrator
    start_time = time.time()
    resp = call_function("orchestrators/pdf_orchestrator", payload)
    status_uri = resp.json()["statusQueryGetUri"]
    
    # Poll until complete
    final_status = "Unknown"
    deadline = time.time() + TIMEOUT
    while time.time() < deadline:
        time.sleep(POLL_INTERVAL)
        status_resp = requests.get(status_uri, timeout=30).json()
        runtime_status = status_resp.get("runtimeStatus", "Unknown")
        custom_status = status_resp.get("customStatus", "")
        
        clear_output(wait=True)
        elapsed = time.time() - start_time
        print(f"[{i+1}/{len(manifest_df)}] {row['filename']} ({row['pages']}pg) | "
              f"{runtime_status} | {custom_status} | {elapsed:.0f}s")
        
        if runtime_status in ("Completed", "Failed", "Terminated"):
            final_status = runtime_status
            break
    
    end_time = time.time()
    duration = end_time - start_time
    
    results.append({
        "filename": row["filename"],
        "pages": row["pages"],
        "size_mb": row["size_mb"],
        "status": final_status,
        "duration_sec": round(duration, 1),
        "blob_prefix": blob_name.rsplit("/", 1)[0] + "/" + row["filename"].replace(".pdf", "")
    })
    
    print(f"  => {final_status} in {duration:.1f}s")

clear_output(wait=True)
print("Benchmark complete!\n")
results_df = pd.DataFrame(results)
display(results_df[["filename", "pages", "size_mb", "status", "duration_sec"]])

## Step 4 — Count OCR Tokens from Extracts

Downloads the extract JSON blobs for each document, concatenates the `content` fields, and counts tokens using `tiktoken` (`cl100k_base` encoding, matching the embedding model tokenizer).

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")
extract_client = blob_service.get_container_client(EXTRACT_CONTAINER)

token_results = []

for _, row in results_df.iterrows():
    if row["status"] != "Completed":
        token_results.append({"filename": row["filename"], "total_tokens": 0, "total_chars": 0, "chunk_count": 0})
        continue
    
    # Extract blobs are stored under the source file prefix (without .pdf extension)
    prefix = row["blob_prefix"]
    # Try both the exact prefix and the blob_prefix from results
    search_prefix = f"{BLOB_PREFIX}/{row['filename'].replace('.pdf', '')}"
    
    all_content = []
    chunk_count = 0
    
    for blob in extract_client.list_blobs(name_starts_with=search_prefix):
        try:
            data = json.loads(extract_client.download_blob(blob.name).readall())
            if "content" in data:
                all_content.append(data["content"])
                chunk_count += 1
        except Exception as e:
            pass
    
    full_text = " ".join(all_content)
    tokens = enc.encode(full_text)
    
    token_results.append({
        "filename": row["filename"],
        "total_tokens": len(tokens),
        "total_chars": len(full_text),
        "chunk_count": chunk_count
    })
    
    print(f"{row['filename']}: {chunk_count} chunks, {len(tokens):,} tokens, {len(full_text):,} chars")

token_df = pd.DataFrame(token_results)

## Step 5 — Combined Results & Visualization

Merges timing and token data into a single table with derived metrics, then plots the results.

In [ ]:
# Merge timing + token data
combined = results_df.merge(token_df, on="filename")
combined["time_per_page"] = (combined["duration_sec"] / combined["pages"]).round(2)
combined["tokens_per_page"] = (combined["total_tokens"] / combined["pages"]).round(0).astype(int)
combined["tokens_per_sec"] = (combined["total_tokens"] / combined["duration_sec"]).round(0).astype(int)

# Display combined table
print("=" * 80)
print("BENCHMARK RESULTS")
print("=" * 80)
display(combined[[
    "filename", "pages", "size_mb", "duration_sec", 
    "total_tokens", "tokens_per_page", "time_per_page", "tokens_per_sec", "status"
]].sort_values("pages"))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_data = combined.sort_values("pages")

# Processing time vs pages
axes[0].bar(plot_data["filename"].str.replace(".pdf", ""), plot_data["duration_sec"], color="steelblue")
axes[0].set_title("Processing Time by Document")
axes[0].set_ylabel("Seconds")
axes[0].tick_params(axis="x", rotation=45)

# Token count vs pages
axes[1].bar(plot_data["filename"].str.replace(".pdf", ""), plot_data["total_tokens"], color="coral")
axes[1].set_title("OCR Token Count by Document")
axes[1].set_ylabel("Tokens (cl100k_base)")
axes[1].tick_params(axis="x", rotation=45)

# Time per page
axes[2].bar(plot_data["filename"].str.replace(".pdf", ""), plot_data["time_per_page"], color="seagreen")
axes[2].set_title("Time per Page")
axes[2].set_ylabel("Seconds / Page")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(f"benchmark-{BENCHMARK_ID}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nChart saved: benchmark-{BENCHMARK_ID}.png")

## Summary

Key metrics to evaluate:
- **Time per page** — indicates pipeline efficiency and whether it scales linearly
- **Tokens per page** — indicates content density; useful for estimating embedding costs
- **Tokens per second** — overall throughput metric

The `automatically_delete` flag was set to `False` so extract blobs remain in the `pdf-extract` container for inspection. Clean them up manually or re-run with `True` when done.